<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/09_Ensemble_Margin_Classification/01_Cherry_Picker_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Margin-Bleed Classification & Ensemble Arbitration

## The Business Problem: The "Cherry-Picker" Parasite Basket
Retailers use deep discounts (loss leaders) to drive foot traffic, assuming customers will fill the rest of their baskets with high-margin items. However, "cherry-pickers" exploit this by purchasing *only* the deeply discounted items. These transactions are parasitic, they generate negative margins and actively bleed profitability.

Our goal is to build a classification system to identify these transactions. Because penalizing a legitimate shopper is dangerous, we will not rely on a single model. We will build a **Hard Voting Ensemble** (Support Vector Machines, K-Nearest Neighbors, and Decision Trees) to act as a strict executive arbiter.

## Step 1: Target Definition
Before classification, we must engineer our target variable (`Y`). We will analyze the historical transactions to calculate the `Discount_Ratio` of every basket. By observing the distribution of these discounts, we will draw a strict mathematical boundary to label baskets as `Parasitic (1)` or `Profitable (0)`.

In [1]:
!pip install completejourney_py
import pandas as pd
import numpy as np
from completejourney_py import get_data

print("Fetching transaction data...")
transactions = get_data()['transactions']

# 1. Group the data to the Basket level
# We want the total sales and total discounts for every unique shopping trip
baskets = transactions.groupby('basket_id').agg(
    Total_Sales_Value=('sales_value', 'sum'),
    Retail_Discount=('retail_disc', 'sum'),
    Coupon_Discount=('coupon_disc', 'sum'),
    Coupon_Match=('coupon_match_disc', 'sum'),
    Total_Items=('quantity', 'sum')
).reset_index()

# 2. Calculate the Absolute Total Discount
# In this dataset, discounts are recorded as negative numbers, so we take the absolute value
baskets['Total_Discount'] = (
    baskets['Retail_Discount'].abs() +
    baskets['Coupon_Discount'].abs() +
    baskets['Coupon_Match'].abs()
)

# 3. Calculate the Gross Value of the basket (what it would have cost without any sales/coupons)
baskets['Gross_Value'] = baskets['Total_Sales_Value'] + baskets['Total_Discount']

# 4. Calculate the Discount Ratio
# We use np.where to avoid division by zero for $0 baskets
baskets['Discount_Ratio'] = np.where(
    baskets['Gross_Value'] > 0,
    baskets['Total_Discount'] / baskets['Gross_Value'],
    0
)

# 5. Let's observe the distribution to make a data-driven decision
print("\n📊 Statistical Distribution of the Discount Ratio across all baskets:")
percentiles = [0.25, 0.50, 0.75, 0.85, 0.90, 0.95, 0.99]
display(baskets['Discount_Ratio'].describe(percentiles=percentiles))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 33.8 MB/s eta 0:00:00
Fetching transaction data...

📊 Statistical Distribution of the Discount Ratio across all baskets:


,Discount_Ratio
count,155848.000000
mean,0.133888
std,0.120011
min,0.000000
25%,0.036643
50%,0.113183
75%,0.202684
85%,0.253880
90%,0.294833
95%,0.365931
